# PlantDoc Data Preprocessing and Logistic Regression Training

Bu notebook, PlantDoc veri setini YOLO formatından klasik makine öğrenmesi için hazırlar ve Logistic Regression modeli eğitir.

PlantDoc dataset'i YOLO formatında etiketlenmiş bir object detection dataset'i. Biz bunu **sınıflandırma (classification)** için kullanacağız.

## Önemli Not

PlantVillage ve PlantDoc için **ayrı ayrı Logistic Regression modelleri** eğitiyoruz çünkü:
- İki dataset'te sınıf isimleri ve ID'ler birebir aynı değil
- Her model kendi label mapping'i ile eğitilmeli
- Bu yaklaşım, Task 2'nin "at least two datasets" şartını temiz şekilde karşılıyor

Adımlar:

1. PlantDoc yolunu ve temel ayarları yapmak
2. YOLO format label dosyalarından sınıf ID'lerini okumak
3. Görselleri 32×32 gri tonlamalı vektörlere dönüştürmek
4. Train/Validation/Test split'lerini yüklemek ve class ID'leri yeniden indekslemek
5. İşlenmiş veriyi `.npy` dosyalarına kaydetmek



## Cell A – Importlar ve PlantDoc yolunu tanımlama


In [1]:
import os
from PIL import Image
import numpy as np

# -------------------
# Configuration
# -------------------
# Root folder of your PlantDoc dataset.
# Expected structure:
# PLANTDOC_ROOT/
#    train/
#       images/
#       labels/
#    valid/
#       images/
#       labels/
#    test/
#       images/
#       labels/
PLANTDOC_ROOT = "../Dataset/plantdoc"  # <-- CHANGE THIS TO YOUR ACTUAL PATH

# Target image size (we will use 32x32 grayscale)
IMG_SIZE = 32


Bu hücrede:

* Gerekli kütüphaneleri yüklüyoruz (`PIL` ile görsel okuma, `numpy` ile veri saklama).
* PlantDoc klasörünün yolunu (`PLANTDOC_ROOT`) ayarlıyoruz.
* Tüm görüntülerin `32×32` gri seviyeye dönüştürüleceğini tanımlıyoruz.


## Cell B – Görüntüyü 32×32 gri vektöre çeviren fonksiyon


In [2]:
def load_image_as_vector(path, img_size=IMG_SIZE):
    """
    Load an image file, convert it to grayscale, resize to img_size x img_size,
    normalize pixel values to [0, 1], and flatten to a 1D list (feature vector).
    """
    with Image.open(path) as img:
        # Convert to grayscale ("L" mode)
        img = img.convert("L")
        # Resize to (img_size, img_size)
        img = img.resize((img_size, img_size))
        # Get pixel values as a flat sequence
        pixels = list(img.getdata())  # length = img_size * img_size
        # Normalize to [0, 1]
        vector = [p / 255.0 for p in pixels]
        return vector


Bu fonksiyon:

1. Dosya yolundan görseli açıyor.
2. Gri tona çeviriyor (`"L"` modu).
3. `IMG_SIZE x IMG_SIZE` boyutuna küçültüyor (32×32).
4. Piksel değerlerini `[0, 255]` aralığından `[0, 1]` aralığına normalize ediyor.
5. 2D görüntüyü **flatten** edip 1D vektör haline getiriyor (1024 boyutlu vektör).

Bu vektör, Logistic Regression modeline girecek **özellik vektörü (feature vector)** olacak.


## Cell C – Label dosyasından class_id okuma (YOLO format)


In [3]:
def read_plantdoc_class_id(label_path):
    """
    Read class id from a YOLO label file.
    YOLO format: class_id x_center y_center width height (normalized 0-1)
    
    If the file has more than one different class id, return None (skip).
    This ensures we have clean classification data (one class per image).
    
    Returns:
        class_id (int) if single class found, None otherwise
    """
    class_ids = set()

    try:
        with open(label_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                class_id = int(parts[0])  # first value is class id
                class_ids.add(class_id)
    except FileNotFoundError:
        return None

    if len(class_ids) == 0:
        return None
    if len(class_ids) > 1:
        # multiple different classes in same image -> ambiguous, skip
        return None

    return class_ids.pop()


Bu fonksiyon:

* PlantDoc label dosyaları YOLO formatında: `class_id x_center y_center width height`
* Biz object detection yapmıyoruz, sadece **sınıf** lazım
* Dosyada birden fazla farklı sınıf varsa (birden fazla tür yaprak vs.) bu görüntüyü **atlıyoruz** → saf bir sınıflandırma seti kurmak için
* Tek sınıf varsa, o class_id'yi döndürüyor


## Cell D – Bir split'i (train/valid/test) yükleyen fonksiyon


In [4]:
def load_plantdoc_split(split_dir, max_images_per_class=None):
    """
    Load a PlantDoc split (train/valid/test) as a classification dataset.

    Args:
        split_dir: path to 'train', 'valid', or 'test' directory that contains:
            images/
            labels/
        max_images_per_class: optional int, limit number of images per class

    Returns:
        X: list[list[float]]  (feature vectors)
        y: list[int]          (original PlantDoc class ids)
    """
    images_dir = os.path.join(split_dir, "images")
    labels_dir = os.path.join(split_dir, "labels")

    X = []
    y = []
    per_class_counts = {}

    for file_name in os.listdir(images_dir):
        lower = file_name.lower()
        if not (lower.endswith(".jpg") or lower.endswith(".jpeg") or lower.endswith(".png")):
            continue

        image_path = os.path.join(images_dir, file_name)

        stem, _ = os.path.splitext(file_name)
        label_path = os.path.join(labels_dir, stem + ".txt")

        class_id = read_plantdoc_class_id(label_path)
        if class_id is None:
            # no label or ambiguous label -> skip
            continue

        if max_images_per_class is not None:
            if per_class_counts.get(class_id, 0) >= max_images_per_class:
                continue

        try:
            x_vec = load_image_as_vector(image_path, img_size=IMG_SIZE)
        except Exception as e:
            print(f"Warning: cannot load {image_path}: {e}")
            continue

        X.append(x_vec)
        y.append(class_id)

        per_class_counts[class_id] = per_class_counts.get(class_id, 0) + 1

    print(
        f"Loaded {len(X)} images from split '{split_dir}' "
        f"for {len(set(y))} classes."
    )
    return X, y


Bu fonksiyon:

* `train/`, `valid/` veya `test/` içindeki `images/` klasöründen resimleri alıyor
* `labels/` klasöründen aynı isimli `.txt` label dosyalarını okuyor
* Her görsel için:
  * `load_image_as_vector` ile 32×32 gri, normalize, flatten hale getiriyor
  * YOLO label'den class_id'yi çıkarıyor
  * Tek sınıf varsa ve limit aşılmamışsa ekliyor
* `y` listesinde **PlantDoc'un orijinal class_id** değerlerini tutuyor (sonra yeniden indekslenecek)


## Cell E – Tüm split'leri yükleme ve class_id'leri yeniden indeksleme


In [5]:
# Paths to train / valid / test directories
train_dir_pd = os.path.join(PLANTDOC_ROOT, "train")
val_dir_pd = os.path.join(PLANTDOC_ROOT, "valid")
test_dir_pd = os.path.join(PLANTDOC_ROOT, "test")

# Load raw splits (with original PlantDoc class ids)
X_pd_train_raw, y_pd_train_raw = load_plantdoc_split(
    train_dir_pd,
    max_images_per_class=None  # you can limit if training is too slow
)

X_pd_val_raw, y_pd_val_raw = load_plantdoc_split(
    val_dir_pd,
    max_images_per_class=None
)

X_pd_test_raw, y_pd_test_raw = load_plantdoc_split(
    test_dir_pd,
    max_images_per_class=None
)

print("Train samples (PlantDoc):", len(X_pd_train_raw))
print("Val samples (PlantDoc):", len(X_pd_val_raw))
print("Test samples (PlantDoc):", len(X_pd_test_raw))

# Collect all class ids from train split (we assume train covers all classes)
unique_ids = sorted(set(y_pd_train_raw))
print("Original PlantDoc class ids:", unique_ids)

# Create mapping: original_id -> new_index (0..C-1)
pd_id_to_idx = {orig_id: i for i, orig_id in enumerate(unique_ids)}
pd_idx_to_id = {i: orig_id for orig_id, i in pd_id_to_idx.items()}

print("Number of PlantDoc classes:", len(unique_ids))
print("Example mapping (orig_id -> new_idx):")
for orig_id, new_idx in list(pd_id_to_idx.items())[:10]:
    print(f"{orig_id} -> {new_idx}")


Loaded 1954 images from split '../Dataset/plantdoc\train' for 28 classes.
Loaded 345 images from split '../Dataset/plantdoc\valid' for 27 classes.
Loaded 238 images from split '../Dataset/plantdoc\test' for 27 classes.
Train samples (PlantDoc): 1954
Val samples (PlantDoc): 345
Test samples (PlantDoc): 238
Original PlantDoc class ids: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 28, 29]
Number of PlantDoc classes: 28
Example mapping (orig_id -> new_idx):
0 -> 0
1 -> 1
2 -> 2
3 -> 3
4 -> 4
5 -> 5
6 -> 6
7 -> 7
8 -> 8
9 -> 9


Bu hücrede:

* Tüm split'leri (train/val/test) yüklüyoruz, her birinde PlantDoc'un orijinal class_id'leri var
* Train split'ten tüm unique class_id'leri topluyoruz (`unique_ids`)
* Bir mapping oluşturuyoruz: `{original_id: 0, original_id: 1, ...}` şeklinde 0'dan başlayan indeksler
* Böylece Logistic Regression modelimiz `0..num_classes-1` aralığında label'lerle çalışabilir


## Cell F – Label'leri yeniden indeksleme fonksiyonu


In [6]:
def reindex_labels(y_list, mapping):
    """
    Map original label ids to new contiguous indices using mapping dict.
    
    Args:
        y_list: list of original class ids
        mapping: dict mapping original_id -> new_index (0..C-1)
    
    Returns:
        y_new: list of reindexed labels
    """
    y_new = []
    skipped = 0
    for label in y_list:
        if label in mapping:
            y_new.append(mapping[label])
        else:
            skipped += 1
    if skipped > 0:
        print(f"Warning: skipped {skipped} samples with unknown class ids.")
    return y_new


# Reindex labels for each split
y_pd_train = reindex_labels(y_pd_train_raw, pd_id_to_idx)
y_pd_val = reindex_labels(y_pd_val_raw, pd_id_to_idx)
y_pd_test = reindex_labels(y_pd_test_raw, pd_id_to_idx)

print("Reindexed Train labels:", len(y_pd_train))
print("Reindexed Val labels:", len(y_pd_val))
print("Reindexed Test labels:", len(y_pd_test))


Reindexed Train labels: 1954
Reindexed Val labels: 345
Reindexed Test labels: 238


Bu fonksiyon ile:

* Her split'in label'lerini mapping kullanarak **0..C-1** aralığına çeviriyoruz
* Artık `y_pd_train`, `y_pd_val`, `y_pd_test` listeleri Logistic Regression için hazır
* Eğer bir label mapping'de yoksa (örneğin test'te yeni bir sınıf), o örneği atlıyoruz


## Cell G – (Opsiyonel) PlantDoc'u `.npy` olarak kaydetme


In [7]:
SAVE_DIR_PD = "preprocessed_plantdoc"
os.makedirs(SAVE_DIR_PD, exist_ok=True)

def save_split_np(X_split, y_split, name_prefix, save_dir):
    """
    Save X and y splits as numpy arrays.
    
    Args:
        X_split: list of feature vectors
        y_split: list of labels (reindexed)
        name_prefix: prefix for filenames (e.g., "train", "val", "test")
        save_dir: directory to save files
    """
    X_array = np.array(X_split, dtype=np.float32)
    y_array = np.array(y_split, dtype=np.int64)
    np.save(os.path.join(save_dir, f"{name_prefix}_X.npy"), X_array)
    np.save(os.path.join(save_dir, f"{name_prefix}_y.npy"), y_array)
    print(f"Saved {name_prefix}_X.npy and {name_prefix}_y.npy to {save_dir}")


save_split_np(X_pd_train_raw, y_pd_train, "train", SAVE_DIR_PD)
save_split_np(X_pd_val_raw, y_pd_val, "val", SAVE_DIR_PD)
save_split_np(X_pd_test_raw, y_pd_test, "test", SAVE_DIR_PD)


Saved train_X.npy and train_y.npy to preprocessed_plantdoc
Saved val_X.npy and val_y.npy to preprocessed_plantdoc
Saved test_X.npy and test_y.npy to preprocessed_plantdoc


Bu hücre:

* İşlenmiş veriyi `preprocessed_plantdoc/` klasörüne `.npy` dosyaları olarak kaydediyor
* PlantVillage'te kullandığın yapının aynısını PlantDoc için de kuruyor:
  * `preprocessed_plantdoc/train_X.npy`, `train_y.npy`
  * `preprocessed_plantdoc/val_X.npy`, `val_y.npy`
  * `preprocessed_plantdoc/test_X.npy`, `test_y.npy`
* Böylece preprocessing adımlarını her seferinde yapmak zorunda kalmazsın
